# 5 · Build the server disk and installer ISO

## Goal

Construct a bootable server RAW disk, compress it, generate payload digests, build an installer UKI, and package the UEFI ISO. The tested builder runs these operations in one **atomic build transaction** to avoid mismatching server payloads and installer checksums.

## Prerequisites

The committed source tree must be clean, the public FreeIPA CA verified out-of-band, the Docker builder available, and `build/output` absent or empty. The build uses privileged Docker.

In [ ]:
from pathlib import Path
import subprocess, sys
START=Path.cwd().resolve()
ROOT=next((p for p in (START,*START.parents) if (p/"myst.yml").is_file() and (p/"scripts/build.sh").is_file()),None)
assert ROOT is not None, "Run from the SC Linux checkout (make book-lab)"
def run(*argv):
    print("+"," ".join(map(str,argv)),flush=True)
    return subprocess.run([str(a) for a in argv],cwd=ROOT,check=True)
print("Source:",ROOT)

## Action 1 · Check that source and output are safe to use

In [ ]:
status=subprocess.run(["git","status","--porcelain","--untracked-files=normal"],cwd=ROOT,capture_output=True,text=True,check=True).stdout
assert not status.strip(), f"Commit source changes before build:\n{status}"
OUTPUT=ROOT/"build/output"
assert not OUTPUT.exists() or not any(OUTPUT.iterdir()), "Choose a fresh output directory; current build/output is not empty"
assert (ROOT/"build/builder.id").is_file(), "Run chapter 4 first"
print("Image destination:",OUTPUT)

## Action 2 · Validate trust before the privileged build

This validates certificate structure and its checked-in digest. It cannot establish your independent verification.

In [ ]:
run("bash","scripts/notebook.sh","notebooks/check.ipynb","--require-trust")

## Action 3 · Construct both images

The builder stages the rootfs, runs `mkosi`, compresses the RAW, writes the payload manifest, builds the installer UKI, assembles the ISO and records provenance. The same step is used in CI.

In [ ]:
run("make","images")

## Action 4 · Inspect the intermediate stages by output

The server is the same payload used by both deployment methods.

In [ ]:
expected={"RAW disk":"raw/sc-linux.raw","Compressed disk":"sc-linux.raw.zst","Installer UKI":"installer/sc-linux-installer.efi","Installer ISO":"sc-linux.iso","Payload manifest":"payload.json"}
for label,rel in expected.items():
    path=OUTPUT/rel
    assert path.is_file() and path.stat().st_size>0, f"Missing {label}: {path}"
    print(f"{label:20} {path.stat().st_size:>14,} B  {rel}")

## Check

The five required outputs exist and are non-empty. This **does not** establish that the RAW or ISO boots.

## Next

[Inspect the artifact checksums and footprint](06-artifacts.ipynb).